In [ ]:
import requests
PORT    = 8081
SERVER_URL = f"http://localhost:{PORT}"   # o https://<id>-{PORT}.cloudspaces.litng.ai

# Health check
resp = requests.get(f"{SERVER_URL}/health")
print("Health:", resp.json())

# Info del modelo
resp = requests.get(f"{SERVER_URL}/model")
print("Model: ", resp.json())

Health: {'status': 'ok', 'model_loaded': True, 'device': 'cuda'}
Model:  {'hub_repo': 'Esteban-Ospina/tweeteval-emotion-bertweet-lora', 'labels': ['anger', 'joy', 'optimism', 'sadness'], 'parameters': '134.9M', 'device': 'cuda'}


In [ ]:
tweet = "I can't believe this is happening, I'm devastated 😢"

resp = requests.post(
    f"{SERVER_URL}/predict",
    json={"text": tweet},
)
resp.raise_for_status()

result = resp.json()
print(f"Tweet: {result['text']}\n")
for p in result["predictions"]:
    bar = "█" * int(p["score"] * 40)
    print(f"  {p['label']:<12} {p['score']:.2%}  {bar}")

Tweet: I can't believe this is happening, I'm devastated 😢

  sadness      46.48%  ██████████████████
  anger        24.07%  █████████
  optimism     16.52%  ██████
  joy          12.93%  █████


In [ ]:
# Predicción en batch
tweets = [
    "I'm so happy today!!! 😊",
    "this is absolutely disgusting #angry #wtf",
    "@user can't believe what just happened... 😢",
    "The future looks bright #hope #goals",
]

resp = requests.post(
    f"{SERVER_URL}/predict/batch",
    json={"texts": tweets},
)
resp.raise_for_status()

print(f"{'Tweet':<50}  {'Emoción':<12}  {'Score':>6}")
print("-" * 74)
for item in resp.json()["results"]:
    top     = item["predictions"][0]
    preview = (item["text"][:47] + "...") if len(item["text"]) > 50 else item["text"].ljust(50)
    print(f"{preview}  {top['label']:<12}  {top['score']:.4f}")

Tweet                                               Emoción        Score
--------------------------------------------------------------------------
I'm so happy today!!! 😊                             joy           0.9269
this is absolutely disgusting #angry #wtf           anger         0.9303
@user can't believe what just happened... 😢         joy           0.4183
The future looks bright #hope #goals                joy           0.8316
